### REST API 호출

In [39]:
import os
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("KAKAO_REST_API_KEY")

if not API_KEY:
    raise RuntimeError("KAKAO_REST_API_KEY is not set")

In [ ]:
import requests

QUERY = "팰리스카운티"  # 바꿔가며 테스트

url = "https://dapi.kakao.com/v2/local/search/keyword.json"
headers = {"Authorization": f"KakaoAK {API_KEY}"}
params = {"query": QUERY}

res = requests.get(url, headers=headers, params=params)
print("HTTP", res.status_code)

if res.status_code == 200:
    places = res.json()["documents"]
    for i, p in enumerate(places[:], 1):
        name = p["place_name"]
        addr = p["road_address_name"] or p["address_name"]
        lat, lng = p["y"], p["x"]  # 위도, 경도
        print(f"{i}. {name} | {addr} | 위도 {lat}, 경도 {lng}")
else:
    print(res.text)  # 오류 내용 확인

HTTP 200
1. 팰리스카운티아파트 | 경기 부천시 원미구 중동로 108 | 위도 37.48991692678554, 경도 126.76761414491631
2. 다이클로 중동팰리스카운티점 | 경기 부천시 원미구 중동로 100 | 위도 37.48865108213565, 경도 126.76582360945835
3. 팰리스카운티작은도서관 | 경기 부천시 원미구 중동로 108 | 위도 37.48961646173878, 경도 126.7678536612872
4. 팰리스카운티아파트 관리사무소 | 경기 부천시 원미구 중동로 108 | 위도 37.48960744063376, 경도 126.76784803551806
5. 팰리스카운티아파트 정문 | 경기 부천시 원미구 중동 1289 | 위도 37.4907794829651, 경도 126.766392519104
6. 팰리스카운티아파트 상가동 | 경기 부천시 원미구 장말로216번길 3 | 위도 37.4923612585226, 경도 126.766195360395
7. 팰리스카운티 테니스장 | 경기 부천시 원미구 중동로 108 | 위도 37.4884178939678, 경도 126.769577194266
8. 크린토피아 코인워시 부천중동팰리스카운티점 | 경기 부천시 원미구 부일로 320 | 위도 37.4882584719073, 경도 126.765484489313
9. GS25 부천팰리스점 | 경기 부천시 원미구 부일로 341 | 위도 37.48849986511326, 경도 126.76818502565204
10. 크린토피아 중동아이파크펠리스점 | 경기 부천시 원미구 장말로216번길 3 | 위도 37.4923577082286, 경도 126.766222510146
11. 팰리스인테리어 | 경기 부천시 원미구 중동로 100 | 위도 37.4887096090609, 경도 126.765804204455
12. 중동팰리스공인중개사사무소 | 경기 부천시 원미구 중동로 108-2 | 위도 37.48997719056374, 경도 126.76801989

In [2]:
API_KEY = "7314ac6f593871935f41e37c6f67590a"
QUERY = "영화관"  # 바꿔가며 테스트

url = "https://dapi.kakao.com/v2/local/search/keyword.json"
headers = {"Authorization": f"KakaoAK {API_KEY}"}
params = {
    "query": QUERY,
    "size": 15,
}

res = requests.get(url, headers=headers, params=params)
# print("HTTP", res.status_code)    

if res.status_code == 200:
    places = res.json()["documents"]

    for p in places:
        print(p["category_name"])

문화,예술 > 영화,영상 > 영화관 > CGV
문화,예술 > 영화,영상 > 영화관 > CGV
문화,예술 > 영화,영상 > 영화관 > CGV
문화,예술 > 영화,영상 > 영화관 > CGV
문화,예술 > 영화,영상 > 영화관 > CGV
문화,예술 > 영화,영상 > 영화관 > CGV
문화,예술 > 영화,영상 > 영화관 > CGV
문화,예술 > 영화,영상 > 영화관 > CGV
문화,예술 > 영화,영상 > 영화관 > CGV
문화,예술 > 영화,영상 > 영화관 > CGV
문화,예술 > 영화,영상 > 영화관 > CGV
문화,예술 > 영화,영상 > 영화관 > 롯데시네마
문화,예술 > 영화,영상 > 영화관 > CGV
문화,예술 > 영화,영상 > 영화관 > CGV
문화,예술 > 영화,영상 > 영화관 > CGV


In [3]:
from pydantic import BaseModel

class Place(BaseModel):
    name: str
    fixed_time: str | None = None

In [4]:
# 거리 구하는 함수

def dist(a, b):
    # a, b = (위도, 경도). 가까운 거리용 간단 버전, 단위 m
    lat1, lng1 = a
    lat2, lng2 = b
    dy = (lat2 - lat1) * 111000   # 위도 1도 ≈ 111km
    dx = (lng2 - lng1) * 88000    # 경도 1도 ≈ 88km (서울 위도 기준)
    return (dx**2 + dy**2) ** 0.5

def walk_min(meters):
    real = meters * 1.3        # 직선거리 -> 실제 걷는 거리 보정
    return real / 67           # 도보 시간(분)

In [5]:
def get_place(place):
    # place: Place (name, fixed_time)
    url = "https://dapi.kakao.com/v2/local/search/keyword.json"
    headers = {"Authorization": f"KakaoAK {API_KEY}"}
    res = requests.get(url, headers=headers, params={"query": place.name})
    docs = res.json()["documents"]
    if not docs:
        return None
    p = docs[0]
    return {
        "name": p["place_name"],
        "lat": float(p["y"]),
        "lng": float(p["x"]),
        "category_name": p["category_name"],
        "fixed_time": place.fixed_time,   # ← 입력에서 그대로 넘김
    }

In [6]:
# 코스 안의 구간별 거리
def hop_distances(route):
    distances = []

    for a, b in zip(route, route[1:]):
        d = dist((a["lat"], a["lng"]), (b["lat"], b["lng"]))
        distances.append(d)

    return distances

# 총 거리
def total_walk_dist(route):
    return sum(hop_distances(route))

In [ ]:
def phase(place):
    name = place["category_name"]

    # 술집류
    if any(w in name for w in ["술집", "호프", "요리주점", "포장마차",
                               "와인바", "칵테일바", "유흥주점", "주류판매"]):
        return "술"

    # 활동: 오래 머무는 체험/관람/놀거리
    if any(w in name for w in [
        # 앉아서 오래 있는 카페형 (카페보다 먼저 걸러야 함)
        "테마카페", "고양이카페", "만화카페", "보드카페", "방탈출카페", "만화방",
        # 관람/체험
        "팝업스토어", "박물관", "전시회", "박람회", "전시관", "미술,공예", "서점",
        # 시간 많이 쓰는 놀거리
        "노래방", "게임방", "PC방",
    ]):
        return "활동"

    # 일반 카페/디저트
    if any(w in name for w in ["카페", "커피전문점", "제과,베이커리",
                               "베이커리", "디저트"]):
        return "카페"

    # 짧게 둘러보는 쇼핑/구경 -> 기타(짧은 체류)
    if any(w in name for w in [
        "뽑기방", "랜덤캡슐토이", "장난감,완구", "취미용품점",
        "액세서리", "생활용품점", "문구", "아트박스", "다이소", "올리브영", "향수",
    ]):
        return "기타"

    # 음식점
    if name.startswith("음식점"):
        return "식사"

    return "기타"

In [8]:
# 장소 이름 리스트를 받아서, 코스 계산에 쓸 수 있는 장소 데이터 리스트로 바꿔주는 함수
# list[str] -> list[place_dict]
def collect(places):              # names -> places
    result = []
    for place in places:
        p = get_place(place)      # name -> place
        if p is None:
            continue
        p["phase"] = phase(p)
        result.append(p)
    return result

In [9]:
from itertools import permutations
import heapq

def best_courses(places, max_count=5, top_n=5):
    # 장소가 하나도 없으면 계산할 코스가 없으므로 빈 리스트 반환
    if len(places) == 0:
        return []

    # 코스에 넣을 장소 개수
    # 예: places가 10개여도 max_count=5면 5개짜리 코스만 만듦
    count = min(len(places), max_count)

    # 거리 기준으로 좋은 후보 top_n개만 보관할 공간
    heap = []

    # heap 안에서 distance가 같은 경우 비교 오류를 피하기 위한 순번
    order = 0

    # places 중 count개를 뽑고, 그 순서까지 모두 고려한 코스를 하나씩 만든다
    # 예: A,B,C 중 2개면 (A,B), (B,A), (A,C), (C,A) ... 모두 나옴
    for course in permutations(places, count):
        # 코스 안에서 각 구간별 거리 리스트
        # 예: [식당->카페 거리, 카페->술집 거리, ...]
        hops = hop_distances(course)

        # 코스 전체 거리 = 구간별 거리의 합
        distance = sum(hops)

        # 나중에 출력하거나 스크리닝할 때 쓰기 좋게 코스 정보를 딕셔너리로 묶음
        item = {
            "course": list(course),          # 실제 장소 순서
            "distance": distance,            # 총 직선거리
            "walk_min": walk_min(distance),  # 예상 도보 시간
            "hops": hops,                    # 구간별 거리
        }

        # heapq는 기본적으로 "가장 작은 값"을 먼저 본다.
        # 우리는 top_n개 중 "가장 나쁜 후보", 즉 거리가 가장 긴 후보를 빨리 꺼내고 싶어서
        # distance에 마이너스를 붙여 저장한다.
        entry = (-distance, order, item)
        order += 1

        # 아직 후보가 top_n개보다 적으면 일단 넣는다
        if len(heap) < top_n:
            heapq.heappush(heap, entry)

        else:
            # 현재 heap 안에 있는 후보 중 가장 나쁜 후보의 거리
            # heap[0][0]은 -distance이므로 다시 마이너스를 붙여 원래 거리로 돌린다
            worst_d = -heap[0][0]

            # 새 코스가 현재 가장 나쁜 후보보다 짧으면 교체한다
            if distance < worst_d:
                heapq.heapreplace(heap, entry)

    # heap에는 entry 형태로 들어 있으므로, 그 안의 item만 꺼낸다
    results = [entry[2] for entry in heap]

    # 최종 결과는 거리 짧은 순서대로 정렬해서 보기 좋게 반환한다
    results = sorted(results, key=lambda x: x["distance"])

    return results

In [10]:
names = [
    "저스트텐동 연남본점",
    "커피 리브레 연남점",
    "연주방",
    "랜디스도넛 연남점",
    "테일러커피 연남점",
    "소이연남",
    "툭툭누들타이",
    "미쁘동 연남점",
    "사루카메",
    "오브젝트 서교점",
]

places = [Place(name=n) for n in names]   # 문자열 -> Place
route = collect(places)

print("찾은 장소 개수:", len(route))
for p in route:
    print(p["phase"], p["name"])

찾은 장소 개수: 10
식사 저스트텐동 연남본점
카페 커피 리브레 연남점
술 연주방
식사 랜디스도넛 연남점
카페 테일러커피 연남1호점
식사 소이연남
식사 티티그릴
식사 미쁘동
식사 사루카메
활동 오브젝트 서교점


In [11]:
def show_courses(top_courses):
    for i, item in enumerate(top_courses, 1):
        course = item["course"]
        hops = item["hops"]

        print()
        print("====", i, "위 ====")
        print("총 직선거리:", round(item["distance"], 1), "m")
        print("예상 도보 시간:", round(item["walk_min"], 1), "분")

        if hops:
            print("가장 긴 구간:", round(max(hops), 1), "m")

        print("코스:")
        for p in course:
            print("-", p["phase"], p["name"])

        print("구간:")
        for a, b, d in zip(course, course[1:], hops):
            print(" ", a["name"], "→", b["name"], round(d, 1), "m")

In [12]:
top_courses = best_courses(route, max_count=5, top_n=5)
show_courses(top_courses)


==== 1 위 ====
총 직선거리: 294.8 m
예상 도보 시간: 5.7 분
가장 긴 구간: 101.1 m
코스:
- 카페 테일러커피 연남1호점
- 식사 소이연남
- 식사 저스트텐동 연남본점
- 술 연주방
- 식사 미쁘동
구간:
  테일러커피 연남1호점 → 소이연남 101.1 m
  소이연남 → 저스트텐동 연남본점 79.0 m
  저스트텐동 연남본점 → 연주방 16.8 m
  연주방 → 미쁘동 97.8 m

==== 2 위 ====
총 직선거리: 294.8 m
예상 도보 시간: 5.7 분
가장 긴 구간: 101.1 m
코스:
- 식사 미쁘동
- 술 연주방
- 식사 저스트텐동 연남본점
- 식사 소이연남
- 카페 테일러커피 연남1호점
구간:
  미쁘동 → 연주방 97.8 m
  연주방 → 저스트텐동 연남본점 16.8 m
  저스트텐동 연남본점 → 소이연남 79.0 m
  소이연남 → 테일러커피 연남1호점 101.1 m

==== 3 위 ====
총 직선거리: 304.4 m
예상 도보 시간: 5.9 분
가장 긴 구간: 110.8 m
코스:
- 카페 테일러커피 연남1호점
- 식사 미쁘동
- 술 연주방
- 식사 저스트텐동 연남본점
- 식사 소이연남
구간:
  테일러커피 연남1호점 → 미쁘동 110.8 m
  미쁘동 → 연주방 97.8 m
  연주방 → 저스트텐동 연남본점 16.8 m
  저스트텐동 연남본점 → 소이연남 79.0 m

==== 4 위 ====
총 직선거리: 304.4 m
예상 도보 시간: 5.9 분
가장 긴 구간: 110.8 m
코스:
- 식사 소이연남
- 식사 저스트텐동 연남본점
- 술 연주방
- 식사 미쁘동
- 카페 테일러커피 연남1호점
구간:
  소이연남 → 저스트텐동 연남본점 79.0 m
  저스트텐동 연남본점 → 연주방 16.8 m
  연주방 → 미쁘동 97.8 m
  미쁘동 → 테일러커피 연남1호점 110.8 m

==== 5 위 ====
총 직선거리: 307.8 m
예상 도보 시간: 6.0 분
가장 긴 구간: 110.8 m
코스:
- 

In [13]:
def has_too_long_hop(item):
    # hops가 비어 있으면, 비교할 구간이 없으므로 "너무 긴 구간 없음"으로 처리
    if not item["hops"]:
        return False

    # 구간별 거리 중 가장 긴 값이 500m를 넘으면 True
    # 즉, 이 코스는 중간에 너무 멀리 걷는 구간이 있다고 판단
    return max(item["hops"]) > 500

In [14]:
def screen_courses(top_courses):
    # 최종 통과한 코스들을 담을 리스트
    passed = []

    # 거리 기준으로 뽑아둔 후보 코스를 하나씩 확인
    for item in top_courses:
        # 코스 안에 500m 넘는 구간이 있으면
        # 이 코스는 제외하고 다음 코스로 넘어감
        if has_too_long_hop(item):
            continue

        # 너무 긴 구간이 없으면 통과 리스트에 추가
        passed.append(item)

    # 통과한 코스들만 반환
    return passed

In [16]:
names = [
    "저스트텐동 연남본점",
    "커피 리브레 연남점",
    "연주방",
    "랜디스도넛 연남점",
    "테일러커피 연남점",
    "소이연남",
    "툭툭누들타이",
    "미쁘동 연남점",
    "사루카메",
    "오브젝트 서교점",
]

places = [Place(name=n) for n in names]   # 문자열 -> Place
route = collect(places)

print("찾은 장소 개수:", len(route))
for p in route:
    print(p["phase"], p["name"])

top_courses = best_courses(route, max_count=5, top_n=5)
passed_courses = screen_courses(top_courses)
show_courses(passed_courses)

찾은 장소 개수: 10
식사 저스트텐동 연남본점
카페 커피 리브레 연남점
술 연주방
식사 랜디스도넛 연남점
카페 테일러커피 연남1호점
식사 소이연남
식사 티티그릴
식사 미쁘동
식사 사루카메
활동 오브젝트 서교점

==== 1 위 ====
총 직선거리: 294.8 m
예상 도보 시간: 5.7 분
가장 긴 구간: 101.1 m
코스:
- 카페 테일러커피 연남1호점
- 식사 소이연남
- 식사 저스트텐동 연남본점
- 술 연주방
- 식사 미쁘동
구간:
  테일러커피 연남1호점 → 소이연남 101.1 m
  소이연남 → 저스트텐동 연남본점 79.0 m
  저스트텐동 연남본점 → 연주방 16.8 m
  연주방 → 미쁘동 97.8 m

==== 2 위 ====
총 직선거리: 294.8 m
예상 도보 시간: 5.7 분
가장 긴 구간: 101.1 m
코스:
- 식사 미쁘동
- 술 연주방
- 식사 저스트텐동 연남본점
- 식사 소이연남
- 카페 테일러커피 연남1호점
구간:
  미쁘동 → 연주방 97.8 m
  연주방 → 저스트텐동 연남본점 16.8 m
  저스트텐동 연남본점 → 소이연남 79.0 m
  소이연남 → 테일러커피 연남1호점 101.1 m

==== 3 위 ====
총 직선거리: 304.4 m
예상 도보 시간: 5.9 분
가장 긴 구간: 110.8 m
코스:
- 카페 테일러커피 연남1호점
- 식사 미쁘동
- 술 연주방
- 식사 저스트텐동 연남본점
- 식사 소이연남
구간:
  테일러커피 연남1호점 → 미쁘동 110.8 m
  미쁘동 → 연주방 97.8 m
  연주방 → 저스트텐동 연남본점 16.8 m
  저스트텐동 연남본점 → 소이연남 79.0 m

==== 4 위 ====
총 직선거리: 304.4 m
예상 도보 시간: 5.9 분
가장 긴 구간: 110.8 m
코스:
- 식사 소이연남
- 식사 저스트텐동 연남본점
- 술 연주방
- 식사 미쁘동
- 카페 테일러커피 연남1호점
구간:
  소이연남 → 저스트텐동 연남본점 79.0 m
  저스트텐동 연남본점 → 연주방 16.8 m

In [29]:
# 활동은 연속 가능
# 카페는 연속이면 어색함
# 식사는 연속이면 어색함
# 술은 식사보다 앞이면 어색함
# 술 다음 카페는 허용 가능
# 술이 꼭 마지막일 필요는 없음

def course_penalty(item, all_places):
    course = item["course"]
    phases = [p["phase"] for p in course]
    all_phases = [p["phase"] for p in all_places]

    penalty = 0

    # 1. 구간이 너무 길면 소프트 패널티
    # 예: 한 구간이 500m 넘으면 탈락이 아니라 점수만 나빠짐
    for hop in item["hops"]:
        if hop > 500:
            penalty += 500

    # 2. 식사 연속 / 카페 연속은 조금 어색함
    for a, b in zip(phases, phases[1:]):
        if a == b and a in ["식사", "카페"]:
            penalty += 300

    # 3. 술이 식사보다 먼저 나오면 어색함
    if "술" in phases and "식사" in phases:
        drink_i = phases.index("술")
        meal_i = phases.index("식사")

        if drink_i < meal_i:
            penalty += 700

    # 4. 전체 후보에 식사가 있는데, 이 코스에 식사가 없으면 큰 패널티
    if "식사" in all_phases and "식사" not in phases:
        penalty += 2000

    # 5. 전체 후보에 카페가 있는데, 이 코스에 카페가 없으면 큰 패널티
    if "카페" in all_phases and "카페" not in phases:
        penalty += 2000

    if "술" in phases:
        drink_i = phases.index("술")
        after_drink = phases[drink_i + 1:]

        if "식사" in after_drink:
            penalty += 1500

        if "카페" in after_drink:
            penalty += 500

    return penalty

In [30]:
def rank_courses(courses, all_places):
    ranked = []

    for item in courses:
        penalty = course_penalty(item, all_places)
        score = item["distance"] + penalty

        item["penalty"] = penalty
        item["score"] = score

        ranked.append(item)

    return sorted(ranked, key=lambda x: x["score"])

In [31]:
top_courses = best_courses(route, max_count=5, top_n=50)

ranked_courses = rank_courses(top_courses, route)

show_courses(ranked_courses[:5])


==== 1 위 ====
총 직선거리: 307.8 m
예상 도보 시간: 6.0 분
가장 긴 구간: 110.8 m
코스:
- 식사 미쁘동
- 카페 테일러커피 연남1호점
- 식사 소이연남
- 식사 저스트텐동 연남본점
- 술 연주방
구간:
  미쁘동 → 테일러커피 연남1호점 110.8 m
  테일러커피 연남1호점 → 소이연남 101.1 m
  소이연남 → 저스트텐동 연남본점 79.0 m
  저스트텐동 연남본점 → 연주방 16.8 m

==== 2 위 ====
총 직선거리: 343.2 m
예상 도보 시간: 6.7 분
가장 긴 구간: 114.4 m
코스:
- 식사 소이연남
- 카페 테일러커피 연남1호점
- 식사 미쁘동
- 식사 저스트텐동 연남본점
- 술 연주방
구간:
  소이연남 → 테일러커피 연남1호점 101.1 m
  테일러커피 연남1호점 → 미쁘동 110.8 m
  미쁘동 → 저스트텐동 연남본점 114.4 m
  저스트텐동 연남본점 → 연주방 16.8 m

==== 3 위 ====
총 직선거리: 387.2 m
예상 도보 시간: 7.5 분
가장 긴 구간: 110.8 m
코스:
- 식사 미쁘동
- 카페 테일러커피 연남1호점
- 식사 저스트텐동 연남본점
- 식사 소이연남
- 술 연주방
구간:
  미쁘동 → 테일러커피 연남1호점 110.8 m
  테일러커피 연남1호점 → 저스트텐동 연남본점 106.1 m
  저스트텐동 연남본점 → 소이연남 79.0 m
  소이연남 → 연주방 91.2 m

==== 4 위 ====
총 직선거리: 388.7 m
예상 도보 시간: 7.5 분
가장 긴 구간: 110.8 m
코스:
- 식사 저스트텐동 연남본점
- 식사 소이연남
- 카페 테일러커피 연남1호점
- 식사 미쁘동
- 술 연주방
구간:
  저스트텐동 연남본점 → 소이연남 79.0 m
  소이연남 → 테일러커피 연남1호점 101.1 m
  테일러커피 연남1호점 → 미쁘동 110.8 m
  미쁘동 → 연주방 97.8 m

==== 5 위 ====
총 직선거리: 378.6 m
예상 도보 시간

In [32]:
from datetime import datetime, timedelta

# phase별 머무는 시간(분)
DWELL_MIN = {"식사": 60, "카페": 60, "술": 90, "활동": 80, "기타": 30}

def dwell_min(place):
    return DWELL_MIN.get(place["phase"], 30)

In [33]:
def schedule_course(item, start="18:00"):
    course = item["course"]
    hops = item["hops"]

    # ① 출발=0분 기준으로 각 장소 '도착 분' 계산
    arrive_min = []
    t = 0
    for i, p in enumerate(course):
        arrive_min.append(t)            # 이 장소 도착(출발 후 t분)
        t += dwell_min(p)               # 머무는 시간만큼 지나고
        if i < len(hops):
            t += walk_min(hops[i])      # 다음 장소까지 걷는 시간만큼 지남

    # ② 진짜 출발 시각 정하기
    anchor = next((i for i, p in enumerate(course) if p.get("fixed_time")), None)
    if anchor is not None:
        # 예약 장소가 'arrive_min[anchor]'분 뒤에 도착하니까, 예약시각에서 그만큼 빼면 출발시각
        fixed = datetime.strptime(course[anchor]["fixed_time"], "%H:%M")
        start_dt = fixed - timedelta(minutes=arrive_min[anchor])
    else:
        start_dt = datetime.strptime(start, "%H:%M")

    # ③ 분 -> 실제 시각
    plan = []
    for p, m in zip(course, arrive_min):
        arrive = start_dt + timedelta(minutes=m)
        leave = arrive + timedelta(minutes=dwell_min(p))
        plan.append({"place": p, "arrive": arrive, "leave": leave})

    item["plan"] = plan
    return item

In [34]:
def show_schedule(item):
    for step in item["plan"]:
        p = step["place"]
        a = step["arrive"].strftime("%H:%M")
        l = step["leave"].strftime("%H:%M")
        mark = " ⏰예약" if p.get("fixed_time") else ""
        print(f"{a}~{l}  {p['phase']} {p['name']}{mark}")

In [35]:
test_places = [
    Place(name="저스트텐동 연남본점"),         # 식사
    Place(name="테일러커피 연남점"),            # 카페
    Place(name="소이연남"),                    # 식사
    Place(name="미쁘동 연남점"),               # 식사
    Place(name="연주방", fixed_time="19:00"),   # 술 + 예약 ⏰
    Place(name="오브젝트 서교점"),             # 구경류 (활동/기타 분류 확인용)
]

route = collect(test_places)

print("찾은 장소:", len(route))
for p in route:
    print(p["phase"], "|", p["name"], "| 예약:", p["fixed_time"], "|", p["category_name"])

ranked = rank_courses(best_courses(route, max_count=5, top_n=50), route)
best = schedule_course(ranked[0])     # 연주방 19:00 기준으로 출발 역산
print()
show_schedule(best)

찾은 장소: 6
식사 | 저스트텐동 연남본점 | 예약: None | 음식점 > 일식
카페 | 테일러커피 연남1호점 | 예약: None | 음식점 > 카페
식사 | 소이연남 | 예약: None | 음식점 > 아시아음식 > 동남아음식 > 태국음식
식사 | 미쁘동 | 예약: None | 음식점 > 일식
술 | 연주방 | 예약: 19:00 | 음식점 > 술집 > 호프,요리주점
활동 | 오브젝트 서교점 | 예약: None | 가정,생활 > 생활용품점

14:54~15:54  식사 미쁘동
15:56~16:56  카페 테일러커피 연남1호점
16:58~17:58  식사 소이연남
17:59~18:59  식사 저스트텐동 연남본점
19:00~20:30  술 연주방 ⏰예약


In [36]:
from urllib.parse import quote

def kakao_route_url(item, by="walk"):
    course = item["course"]
    if len(course) < 2:
        return None

    def point(p):
        name = quote(p["name"], safe="")
        return f"{name},{p['lat']},{p['lng']}"

    points = "/".join(point(p) for p in course)
    return f"https://map.kakao.com/link/by/{by}/{points}"

In [37]:
print(kakao_route_url(best))
# kakaomap://route?sp=37.562...,126.925...&ep=...&vp=...&vp2=...&by=foot

https://map.kakao.com/link/by/walk/%EB%AF%B8%EC%81%98%EB%8F%99,37.5621016631213,126.926190830908/%ED%85%8C%EC%9D%BC%EB%9F%AC%EC%BB%A4%ED%94%BC%20%EC%97%B0%EB%82%A81%ED%98%B8%EC%A0%90,37.563085683374,126.926402647715/%EC%86%8C%EC%9D%B4%EC%97%B0%EB%82%A8,37.5635139207821,126.92538808404/%EC%A0%80%EC%8A%A4%ED%8A%B8%ED%85%90%EB%8F%99%20%EC%97%B0%EB%82%A8%EB%B3%B8%EC%A0%90,37.562811054636846,126.92524730440957/%EC%97%B0%EC%A3%BC%EB%B0%A9,37.5626921983887,126.925366266672
